# Kang PBMC — end-to-end pyvae sanity check

An end-to-end run of pyvae on the Kang 2018 PBMC dataset (control vs interferon-β stimulated).

The goal is to verify that pyvae's modern training loop (`train_ivae_modern`) and interpretation stack (`bayes_factor_da`, `integrated_gradients`, `predict_counterfactual`) recover the expected interferon-response biology.

Sections:
1. Setup and data loading
2. Reactome adjacency and covariate matrix
3. Model construction and training
4. UMAP of pathway activations
5. Differential module ranking (Bayes factor + Wilcoxon)
6. Gene-level attribution via Integrated Gradients
7. Conditioning and counterfactual prediction
8. Takeaways

## 1. Setup and data loading

In [1]:
from __future__ import annotations

from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import torch

import pyvae
from pyvae import (
    InformedVAE,
    bayes_factor_da,
    build_model_config,
    integrated_gradients,
    load_kang,
    set_all_seeds,
    sync_gexp_adj,
    train_ivae_modern,
)

# Deterministic runs
SEED = 42
set_all_seeds(SEED)

# Where to store the downloaded Kang data (git-ignored)
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

from importlib.metadata import version as _pkg_version

def _try_version(name):
    try:
        return _pkg_version(name)
    except Exception:
        return "(unavailable)"

print(f"pyvae version: {_try_version('pyvae')}")
print(f"torch version: {torch.__version__}")
print(f"scanpy version: {_try_version('scanpy')}")

pyvae version: 0.2.0
torch version: 2.12.1
scanpy version: 1.12.1


### Load the dataset

`load_kang` downloads the AnnData from figshare on first call, then caches it locally. It performs standard preprocessing:

- Normalizes to a fixed library size and applies `log1p`
- Selects highly variable genes (default: top 2000)
- Preserves raw counts in `adata.layers["counts"]` for the negative-binomial likelihood
- Maps the raw condition labels (`ctrl` / `stim`) to `control` / `stimulated`

In [2]:
adata = load_kang(data_folder=str(DATA_DIR), n_genes=2000)
adata

/Users/amiraynede/Thesis/pyvae/pyvae/datasets.py:68: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  adata.obs["label"] = adata.obs["label"].replace(
OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


AnnData object with n_obs × n_vars = 24673 × 2000
    obs: 'nCount_RNA', 'nFeature_RNA', 'tsne1', 'tsne2', 'condition', 'cluster', 'cell_type', 'replicate', 'nCount_SCT', 'nFeature_SCT', 'integrated_snn_res.0.4', 'seurat_clusters'
    var: 'name', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'log1p', 'hvg'
    obsm: 'X_pca', 'X_umap'
    layers: 'counts'

### Sanity check

Compare cell counts against Gundogdu et al. (2023), Table 1:

- Control: 1316 B, 2932 CD14+ Mono, 5560 CD4 T, 811 CD8 T, 258 DC, 520 FCGR3A+ Mono, 63 Megakaryocytes, 855 NK
- Stimulated: 1335 B, 2765 CD14+ Mono, 5678 CD4 T, 810 CD8 T, 271 DC, 569 FCGR3A+ Mono, 69 Megakaryocytes, 861 NK

Exact numbers will differ slightly (different HVG selection, filtering choices, scanpy versions), but the order of magnitude and overall structure should match.

In [3]:
print(f"AnnData shape: {adata.shape}   (cells x genes)")
print(f"n_cells: {adata.n_obs:,}")
print(f"n_genes (HVG): {adata.n_vars:,}")
print()
print("Available layers:", list(adata.layers.keys()))
print("Available obs columns:", list(adata.obs.columns))
print()

# Distribution by condition and cell type
xtab = pd.crosstab(adata.obs["cell_type"], adata.obs["condition"])
print("Cells by type x condition:")
print(xtab)
print()
print(f"Total control:    {(adata.obs['condition'] == 'control').sum():,}")
print(f"Total stimulated: {(adata.obs['condition'] == 'stimulated').sum():,}")

AnnData shape: (24673, 2000)   (cells x genes)
n_cells: 24,673
n_genes (HVG): 2,000

Available layers: ['counts']
Available obs columns: ['nCount_RNA', 'nFeature_RNA', 'tsne1', 'tsne2', 'condition', 'cluster', 'cell_type', 'replicate', 'nCount_SCT', 'nFeature_SCT', 'integrated_snn_res.0.4', 'seurat_clusters']

Cells by type x condition:
condition          control  stimulated
cell_type                             
CD4 T cells           5560        5678
CD14+ Monocytes       2932        2765
B cells               1316        1335
NK cells               855         861
CD8 T cells            811         810
FCGR3A+ Monocytes      520         569
Dendritic cells        258         271
Megakaryocytes          63          69

Total control:    12,315
Total stimulated: 12,358
